# Preparation carbon data for blue run: 20260515_cdgvm

In [ ]:
# carbon: cdgvm
# secondary, pasture, cropland: ratios from default

In [ ]:
# Define run identifier (to be used for naming files and folders of blue run)
fstr = '20260515_cdgvm'

---

In [ ]:
# Libraries
import os, shutil
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'
dir05 = '../paper_ecosystem_carbon/output/05_rf_constrain/'
dir06 = '../paper_ecosystem_carbon/output/06_blue_prep/'
dir_blue = '../paper_ecosystem_carbon/blue_input/'

---

### Pre-processing

In [ ]:
# Get carbon density data
ds_dgvm = xr.open_zarr(dir04 + 'ds_dgvm_rffill_r025.zarr')

# Get potential occurance data
ds_occur = xr.open_zarr(dir02 + 'ds_pot_occur_r025.zarr')

In [ ]:
# Define ds_cin
# Calculate mean across models
ds_cin = ds_dgvm.mean('model')

In [ ]:
# Get blue pft distribution data
ds_pft = xr.open_dataset(dir_blue + 'potveg_tropforestoutside40-corr_ratmax.nc')
ds_pft = ds_pft.reindex(lat=list(reversed(ds_pft.lat)))

# Get blue pft distribution
da_pft = xr.where(ds_pft > 0, True, False).cover_fract
# Blue land mask 
#da_land = xr.where(~np.isnan(ds_pft.cover_fract.sel(type=1)), True, False)

#1	Tropical evergreen forest
#2	Tropical deciduous forest
#3	Temperate evergreen broadleaf forest
#4	Temperate/boreal deciduous broadleaf forest
#5	Temperate/boreal evergreen conifers
#6	Temperate/boreal deciduous conifers
#7	Raingreen shrubs
#8	Summergreen shrubs
#9	C3 natural grasses
#10	C4 natural grasses
#11	Tundra

In [ ]:
# Get blue carbon default data
df_cd = pd.read_csv(dir_blue + 'carbon-and-flux_vals_JP.csv')
list_df_cd_orig_cols = df_cd.columns
df_cd.columns = df_cd.loc[0]
df_cd = df_cd.drop(df_cd.index[0]).reset_index(drop=True)

---

### Prepare pft arrays

In [ ]:
# Translate to blue pfts
def prep_ds_c_pft(ds_cin, ds_occur, ctype):

    # Dataset with rf values assigned to blue pfts
    ds_c_pft = xr.Dataset()
    ds_c_pft['pft1'] = ds_cin[f'{ctype}_trop_ef']
    ds_c_pft['pft2'] = ds_cin[f'{ctype}_trop_df']
    ds_c_pft['pft3'] = ds_cin[f'{ctype}_temp_ebf']
    ds_c_pft['pft4'] = ds_cin[f'{ctype}_temp_dbf']
    ds_c_pft['pft5'] = ds_cin[f'{ctype}_temp_enf']
    ds_c_pft['pft6'] = ds_cin[f'{ctype}_temp_dnf']
    ds_c_pft['pft7'] = ds_cin[f'{ctype}_glob_shrub']
    ds_c_pft['pft8'] = ds_cin[f'{ctype}_glob_shrub']
    ds_c_pft['pft9'] = ds_cin[f'{ctype}_glob_grass']
    ds_c_pft['pft10'] = ds_cin[f'{ctype}_glob_grass']
    ds_c_pft['pft11'] = xr.where(ds_occur['glob_tundr'] == True,
                                   ds_cin[f'{ctype}_glob_tundr'],
                                   ds_cin[f'{ctype}_glob_mount'])
    
    return ds_c_pft

ds_c_pft_cveg = prep_ds_c_pft(ds_cin, ds_occur, 'cveg')
ds_c_pft_csoil = prep_ds_c_pft(ds_cin, ds_occur, 'csoil')

In [ ]:
#cveg
ds_cveg = xr.Dataset()

# Primary and secondary
# Assumption: RF values are primary
# Get secondary, pasture and crop through ratio of default
for i in range(1, 12):
    # Get default values
    v_prim = float(df_cd['Primary - Veg C'].iloc[i-1])
    v_secd = float(df_cd['Secondary - Veg C'].iloc[i-1])
    v_past = float(df_cd['Pasture - Veg C'].iloc[i-1])
    v_crop = float(df_cd['Crop - Veg C'].iloc[i-1])
    # Get ratio of primary to other luc values
    ratio_prim_secd = v_secd / v_prim
    ratio_prim_past = v_past / v_prim
    ratio_prim_crop = v_crop / v_prim
    # Add prim, secd, past, and crop arrays
    ds_cveg[f'cveg_prim_pft{i}'] = ds_c_pft_cveg[f'pft{i}']
    ds_cveg[f'cveg_secd_pft{i}'] = ds_c_pft_cveg[f'pft{i}'] * ratio_prim_secd
    ds_cveg[f'cveg_past_pft{i}'] = ds_c_pft_cveg[f'pft{i}'] * ratio_prim_past
    ds_cveg[f'cveg_crop_pft{i}'] = ds_c_pft_cveg[f'pft{i}'] * ratio_prim_crop

# Limit to area where pft occurs according to blue pft dataset
for l in ['prim', 'secd', 'crop', 'past']:
    for i in range(1, 12):
        v = f'cveg_{l}_pft{i}'
        da_var = ds_cveg[v].where(da_pft.sel(type=i).data)
        ds_cveg[v] = da_var

ds_cveg

In [ ]:
# csoil
ds_csoil = xr.Dataset()

# Primary and secondary
# Assumption: RF values are primary
# Get secondary, pasture and crop through ratio of default
for i in range(1, 12):
    # Get default values
    v_prim = float(df_cd['Primary - Soil C'].iloc[i-1])
    v_secd = float(df_cd['Secondary - Soil C'].iloc[i-1])
    v_past = float(df_cd['Pasture - Soil C'].iloc[i-1])
    v_crop = float(df_cd['Crop - Soil C'].iloc[i-1])
    # Get ratio of primary to other luc values
    ratio_prim_secd = v_secd / v_prim
    ratio_prim_past = v_past / v_prim
    ratio_prim_crop = v_crop / v_prim
    # Add prim, secd, past, and crop arrays
    ds_csoil[f'csoil_prim_pft{i}'] = ds_c_pft_csoil[f'pft{i}']
    ds_csoil[f'csoil_secd_pft{i}'] = ds_c_pft_csoil[f'pft{i}'] * ratio_prim_secd
    ds_csoil[f'csoil_past_pft{i}'] = ds_c_pft_csoil[f'pft{i}'] * ratio_prim_past
    ds_csoil[f'csoil_crop_pft{i}'] = ds_c_pft_csoil[f'pft{i}'] * ratio_prim_crop

# Limit to area where pft occurs according to blue pft dataset
for l in ['prim', 'secd', 'crop', 'past']:
    for i in range(1, 12):
        v = f'csoil_{l}_pft{i}'
        da_var = ds_csoil[v].where(da_pft.sel(type=i).data)
        ds_csoil[v] = da_var

ds_csoil

In [ ]:
# Export
ds_cveg_csoil = xr.merge([ds_cveg, ds_csoil])

---

### Create blue input files

In [ ]:
# Remove folder and files of carbon data if already exist
if os.path.exists(f'{dir_blue}{fstr}/'): 
    shutil.rmtree(f'{dir_blue}{fstr}/')
# Create folder for carbon data
os.mkdir(f'{dir_blue}{fstr}/')

In [ ]:
# Export carbon data as single datasets and in the right format for blue
for ltype in ['prim', 'secd', 'past', 'crop']:
    for ctype in ['cveg', 'csoil']:
        for npft in range(1,12):
            v = f'{ctype}_{ltype}_pft{npft}'
            ds_cveg_csoil[v] \
                .rename('carbonDensityTHa') \
                .round(0) \
                .fillna(0) \
                .astype('int') \
                .expand_dims(dict(time = [2010])) \
                .to_netcdf(f'{dir_blue}{fstr}/{v}.nc', mode='w')

In [ ]:
# Prepare blue csv file and export
df_cd = pd.read_csv(f'{dir_blue}carbon-and-flux_vals_JP.csv')

list_df_cd_orig_cols = df_cd.columns

df_cd.columns = df_cd.loc[0]
df_cd = df_cd.drop(df_cd.index[0]).reset_index(drop=True)

for i in [1,2,3,4,5,6,7,8,9,10,11]:
    df_cd.loc[i-1, 'Primary - Veg C'] = f'cveg_prim_pft{i}.nc'
    df_cd.loc[i-1, 'Secondary - Veg C'] = f'cveg_secd_pft{i}.nc'
    df_cd.loc[i-1, 'Crop - Veg C'] = f'cveg_crop_pft{i}.nc'
    df_cd.loc[i-1, 'Pasture - Veg C'] = f'cveg_past_pft{i}.nc'

    df_cd.loc[i-1, 'Primary - Soil C'] = f'csoil_prim_pft{i}.nc'
    df_cd.loc[i-1, 'Secondary - Soil C'] = f'csoil_secd_pft{i}.nc'
    df_cd.loc[i-1, 'Crop - Soil C'] = f'csoil_crop_pft{i}.nc'
    df_cd.loc[i-1, 'Pasture - Soil C'] = f'csoil_past_pft{i}.nc'
    
    df_cd.loc[i-1, 'Minimum soil C following harvest'] = f'csoil_crop_pft{i}.nc'
    df_cd.loc[i-1, 'Soil carbon after initial, rapid loss after clearing'] = \
        f'csoil_crop_pft{i}.nc'

df_cd = pd.concat([df_cd.columns.to_frame().T, df_cd], ignore_index=True)
df_cd.columns = list_df_cd_orig_cols

df_cd.to_csv(f'{dir_blue}carbon-and-flux_vals_JP_spatiallyExplicit_{fstr}.csv', 
             index=False)